In [2]:
import networkx as nx
from itertools import combinations
import pandas as pd
import dask.dataframe as ddf

In [ ]:
cdr = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/combined_real_cdr_for_cider.csv'
)
subscribers = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/input_data/togo/togo_full_2018_10/combined_real_subscribers.csv'
)

/var/folders/y5/65jnz7bx5t3_rcnkb0xz6fvw0000gn/T/ipykernel_49062/1779454872.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  cdr = pd.read_csv(


In [24]:

contacts = pd.concat([
    cdr[['caller_msisdn', 'recipient_msisdn']].rename(columns={'caller_msisdn': 'ego', 'recipient_msisdn': 'alter'}),
    cdr[['recipient_msisdn', 'caller_msisdn']].rename(columns={'recipient_msisdn': 'ego', 'caller_msisdn': 'alter'}),
]).drop_duplicates()

alter_to_egos = contacts.dropna(how='any').groupby('alter')['ego'].apply(list)

shared_edges = set()
for egos in alter_to_egos:
    for u, v in combinations(egos, 2):
        shared_edges.add((min(u, v), max(u, v)))

# Build graph
G = nx.Graph()
G.add_nodes_from(contacts['ego'].unique())
G.add_edges_from(shared_edges)

sub_ids = set(subscribers['subscriber_id'])
G_sub = G.subgraph([n for n in G.nodes if n in sub_ids])

print(f"Nodes : {G_sub.number_of_nodes():,}")
print(f"Edges : {G_sub.number_of_edges():,}")
print(f"Density: {nx.density(G_sub):.4f}")

Nodes : 3,924
Edges : 120,516
Density: 0.0157


In [ ]:
total = G_sub.number_of_nodes()
for k in [1, 2, 3, 4, 5, 6, 7, 8, 9, b10, 20]:
    count = sum(1 for n in G_sub.nodes if G_sub.degree(n) >= k)
    print(f"degree >= {k:>3}: {count:,} / {total:,} ({100*count/total:.1f}%)")

degree >=   1: 2,531 / 3,924 (64.5%)
degree >=   2: 1,875 / 3,924 (47.8%)
degree >=   3: 1,524 / 3,924 (38.8%)
degree >=   4: 1,288 / 3,924 (32.8%)
degree >=   5: 1,150 / 3,924 (29.3%)
degree >=   6: 1,062 / 3,924 (27.1%)
degree >=   7: 1,004 / 3,924 (25.6%)
degree >=   8: 959 / 3,924 (24.4%)
degree >=   9: 930 / 3,924 (23.7%)
degree >=  10: 908 / 3,924 (23.1%)
degree >=  20: 809 / 3,924 (20.6%)


In [3]:
from scipy.stats import rankdata
import os
import pandas as pd
import matplotlib.pyplot as plt
import scipy as sp
import numpy as np

# Replace with real data

antennas = pd.read_csv(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/togo_2018_oct_dec/real_data/antennas.csv'
)
existing_features = pd.read_parquet(
    '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/togo_2018_oct_dec/features/*.parquet'
)

# Derive top region per subscriber
region_cols = [c for c in existing_features.columns if 'location_regions_' in c and not c.endswith('_percent')]
region_names = [c.replace('location_regions_', '') for c in region_cols]
ranks = existing_features[region_cols].values.argsort(axis=1)[:, ::-1]
existing_features['top_region_1'] = [region_names[i] for i in ranks[:, 0]]
existing_features['top_region_2'] = [region_names[i] for i in ranks[:, 1]]
existing_features['top_region_3'] = [region_names[i] for i in ranks[:, 2]]

# Drop location columns and flag maritime subscribers
cols_loc = ~existing_features.columns.str.contains('location')
existing_features_reduced = existing_features.loc[:, cols_loc].copy()
existing_features_reduced['top_MARITIME'] = existing_features_reduced['top_region_1'].str.contains('MARITIME').astype(int)

# Split by region and extract activity
maritime = existing_features_reduced[existing_features_reduced['top_MARITIME'] == 1]
non_maritime = existing_features_reduced[existing_features_reduced['top_MARITIME'] == 0]
activity_maritime = maritime['active_days_allweek_allday'].dropna()
activity_non_maritime = non_maritime['active_days_allweek_allday'].dropna()

# Z-score normalize within region
mean_m, sd_m = activity_maritime.mean(), activity_maritime.std()
mean_nm, sd_nm = activity_non_maritime.mean(), activity_non_maritime.std()
normalized_m = (activity_maritime - mean_m) / sd_m
normalized_nm = (activity_non_maritime - mean_nm) / sd_nm

# Quantile normalize within region
def quantile_normalize(series):
    r = rankdata(series.values, method='average')
    return (r - 1) / (len(r) - 1)

qnorm_m = quantile_normalize(activity_maritime)
qnorm_nm = quantile_normalize(activity_non_maritime)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(activity_non_maritime, alpha=0.5, density=True, label='non-maritime')
axes[0].hist(activity_maritime, alpha=0.5, density=True, label='maritime')
axes[0].set_title('Raw')
axes[0].legend()

axes[1].hist(normalized_nm, alpha=0.5, density=True, label='non-maritime')
axes[1].hist(normalized_m, alpha=0.5, density=True, label='maritime')
axes[1].set_title('Z-score (regional)')
axes[1].legend()

axes[2].hist(qnorm_nm, alpha=0.5, density=True, label='non-maritime')
axes[2].hist(qnorm_m, alpha=0.5, density=True, label='maritime')
axes[2].set_title('Quantile (regional)')
axes[2].legend()

plt.suptitle('active_days_allweek_allday: Maritime vs Non-Maritime', y=1.02)
plt.tight_layout()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket/togo_2018_oct_dec/features/*.parquet'